## Task 1: Create SQLite Database

In [1]:
import sqlite3

# Create SQLite database
connection = sqlite3.connect('company.db')

# Create cursor
cursor = connection.cursor()

# Employees table
cursor.execute("""
CREATE TABLE employees(
    id INTEGER PRIMARY KEY, name TEXT, department TEXT, salary INTEGER
    )
""")

print("Employees table created!")

# Sales table
cursor.execute("""
CREATE TABLE sales(
    sale_id INTEGER PRIMARY KEY, employee_id INTEGER, amount INTEGER, sale_date TEXT
    )
""")

print("Sales table created!")

Employees table created!
Sales table created!


## Task 2: Insert Sample Data

In [2]:

connection = sqlite3.connect("company.db")
cursor = connection.cursor()

# Insert 10 employees
employees = [
    (1, "Amit", "IT", 80000),
    (2, "Priya", "HR", 60000),
    (3, "Rahul", "Sales", 55000),
    (4, "Neha", "IT", 90000),
    (5, "Vikas", "Finance", 75000),
    (6, "Sneha", "HR", 65000),
    (7, "Arjun", "Sales", 70000),
    (8, "Pooja", "Finance", 85000),
    (9, "Karan", "IT", 95000),
    (10, "Anjali", "Sales", 60000)
]

cursor.executemany("""
INSERT INTO employees (id, name, department, salary)
VALUES (?,?,?,?)
""", employees
)

# Insert 10 sales records
sales = [
    (1, 1, 15000, "2026-01-05"),
    (2, 2, 12000, "2026-01-08"),
    (3, 3, 18000, "2026-01-10"),
    (4, 4, 25000, "2026-01-12"),
    (5, 5, 22000, "2026-01-15"),
    (6, 6, 14000, "2026-01-18"),
    (7, 7, 30000, "2026-01-20"),
    (8, 8, 27000, "2026-01-22"),
    (9, 9, 35000, "2026-01-25"),
    (10, 10, 20000, "2026-01-28")
]

cursor.executemany("""
INSERT INTO sales (sale_id, employee_id, amount, sale_date)
VALUES (?, ?, ?, ?)
""", sales)

connection.commit()

# Verify inserted records
employee_data = cursor.execute("""SELECT * FROM employees""")
for employee in employee_data:
    print(employee)

sales_data = cursor.execute("""SELECT * FROM sales""")
for sale in sales_data:
    print(sale)

connection.close()

(1, 'Amit', 'IT', 80000)
(2, 'Priya', 'HR', 60000)
(3, 'Rahul', 'Sales', 55000)
(4, 'Neha', 'IT', 90000)
(5, 'Vikas', 'Finance', 75000)
(6, 'Sneha', 'HR', 65000)
(7, 'Arjun', 'Sales', 70000)
(8, 'Pooja', 'Finance', 85000)
(9, 'Karan', 'IT', 95000)
(10, 'Anjali', 'Sales', 60000)
(1, 1, 15000, '2026-01-05')
(2, 2, 12000, '2026-01-08')
(3, 3, 18000, '2026-01-10')
(4, 4, 25000, '2026-01-12')
(5, 5, 22000, '2026-01-15')
(6, 6, 14000, '2026-01-18')
(7, 7, 30000, '2026-01-20')
(8, 8, 27000, '2026-01-22')
(9, 9, 35000, '2026-01-25')
(10, 10, 20000, '2026-01-28')


## Task 3: Create SQLAlchemy Engine

In [5]:
from sqlalchemy import create_engine, inspect

# Create SQLAlchemy engine
engine = create_engine('sqlite:///company.db')

with engine.connect() as connection:
    print('Database connected successfully')

tables = inspect(engine).get_table_names()

print("Available tables:", tables)

Database connected successfully
Available tables: ['employees', 'sales']


## Task 4: Create LangChain SQLDatabase Object

In [ ]:
from langchain_community.utilities import SQLDatabase

# Connect LangChain to SQLite database and using prev task engine
db = SQLDatabase(engine)

# Display available tables
print("Tables:", db.get_usable_table_names())

# Display schema information
print(db.get_table_info())

Tables: ['employees', 'sales']

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Amit	IT	80000
2	Priya	HR	60000
3	Rahul	Sales	55000
*/


CREATE TABLE sales (
	sale_id INTEGER, 
	employee_id INTEGER, 
	amount INTEGER, 
	sale_date TEXT, 
	PRIMARY KEY (sale_id)
)

/*
3 rows from sales table:
sale_id	employee_id	amount	sale_date
1	1	15000	2026-01-05
2	2	12000	2026-01-08
3	3	18000	2026-01-10
*/


## Task 5: Initialize SQL Toolkit

In [8]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# Initialize LLM
llm = ChatGroq(model="openai/gpt-oss-120b")

# Create SQL toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Get available tools
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "-", tool.description)

sql_db_query - Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.
sql_db_schema - Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3
sql_db_list_tables - Input is an empty string, output is a comma-separated list of tables in the database.
sql_db_query_checker - Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!


## Task 6: Create SQL Agent

In [10]:
from langchain_community.agent_toolkits import create_sql_agent

agent = create_sql_agent(llm=llm, toolkit=toolkit, verbose=True, agent_type="tool-calling", agent_executor_kwargs={"handle_parsing_errors": True},)

# Test the SQL agent
response = agent.invoke({
    "input": "How many employees are there in each department?"
})

print(response["output"])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


employees, sales
Invoking: `sql_db_schema` with `{'table_names': 'employees'}`



CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Amit	IT	80000
2	Priya	HR	60000
3	Rahul	Sales	55000
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10;'}`


SELECT department, COUNT(*) AS employee_count 
FROM employees 
GROUP BY department 
ORDER BY employee_count DESC 
LIMIT 10;
Invoking: `sql_db_query` with `{'query': 'SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10;'}`


[('Sales', 3), ('IT', 3), ('HR', 2), ('Finance', 2)]**Employees per department**

| Department | Employee Count |
|------------|-----------

## Task 7: Chat with SQL Database

In [11]:
questions = [
    "How many employees are there in each department?",
    "Who has the highest salary?",
    "What is the total sales amount?",
    "What is the average salary per department?"
]

for question in questions:
    print(f"\nQuestion: {question}")
    response = agent.invoke({"input": question})
    print("Answer:", response["output"])


Question: How many employees are there in each department?


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


employees, sales
Invoking: `sql_db_schema` with `{'table_names': 'employees'}`



CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Amit	IT	80000
2	Priya	HR	60000
3	Rahul	Sales	55000
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10;'}`


SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10;
Invoking: `sql_db_query` with `{'query': 'SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC LIMIT 10;'}`


[('Sales', 3), ('IT', 3), ('HR', 2), ('Finance', 2)]Here’s the numb

## Task 8: Handling Ambiguous Queries

In [12]:
# Test ambiguous questions
questions = [
    "Show me the best employees.",
    "Show me the highest amount.",
    "Give me the details."
]

for question in questions:
    print(f"\nQuestion: {question}")
    response = agent.invoke({"input": question})
    print("Answer:", response["output"])


Question: Show me the best employees.


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


employees, sales
Invoking: `sql_db_schema` with `{'table_names': 'employees, sales'}`



CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Amit	IT	80000
2	Priya	HR	60000
3	Rahul	Sales	55000
*/


CREATE TABLE sales (
	sale_id INTEGER, 
	employee_id INTEGER, 
	amount INTEGER, 
	sale_date TEXT, 
	PRIMARY KEY (sale_id)
)

/*
3 rows from sales table:
sale_id	employee_id	amount	sale_date
1	1	15000	2026-01-05
2	2	12000	2026-01-08
3	3	18000	2026-01-10
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT e.name, e.department, e.salary, IFNULL(SUM(s.amount),0) AS total_sales FROM employees e LEFT JOIN sales s ON e.id = s.employee_id GROUP BY e.id, e.name, e.department, e.salary ORDER BY total_sales DESC LIMIT 10;'}`


SELECT e.na

#### Observations & Insights

1. SQL agents understand natural-language questions, generate SQL queries dynamically, select appropriate tools, and retrieve results from databases.


2. SQL agents: Retrieve structured data from databases using SQL queries.

RAG: Retrieves relevant information from documents or vector databases and uses it to generate answers.


3. Risks of allowing unrestricted SQL access

Accidental deletion or modification of data.

Unauthorized access to sensitive information.
